# GPT-2 residual stream — every block, streamed to HDF5

A decoder-only model, capturing all 12 blocks at once: 12 x seq_len x 768 floats
per passage. This is where keeping activations in RAM stops being reasonable and
`path=` starts to matter.

Downloads on first run: WikiText-2 (~5 MB) and GPT-2 weights (~500 MB).

In [ ]:
from pathlib import Path

from datasets import load_dataset
from torch.utils.data import Dataset
from transformers import AutoModel, AutoTokenizer

from nnact import ActivationMapper, H5ActivationStore, Sample

CACHE = Path("gpt2_residual.h5")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 ships without a pad token
model = AutoModel.from_pretrained("gpt2")  # not ...LMHeadModel: no logits needed

In [ ]:
class WikiTextSamples(Dataset[Sample]):
    """WikiText passages, tokenised to a fixed length.

    nnact stacks Sample.data across a batch, so every row must be the same
    shape: pad to a fixed max_length rather than per batch. Attention masks
    are kept on the instance for pooling later.
    """

    def __init__(self, tokenizer, n: int = 256, max_length: int = 64) -> None:
        raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
        # Drop blanks and the "= Heading =" lines the raw dump is full of.
        self.texts = [
            t.strip()
            for t in raw["text"]
            if len(t.strip()) > 120 and not t.strip().startswith("=")
        ][:n]
        self.encoded = tokenizer(
            self.texts,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Sample:
        return Sample(id=f"wiki_{idx:04d}", data=self.encoded["input_ids"][idx])


dataset = WikiTextSamples(tokenizer, n=512, max_length=64)
mapper = ActivationMapper(model)

# The residual stream: every block, plus the final layer norm.
LAYERS = [f"h.{i}" for i in range(12)] + ["ln_f"]
print(f"{len(dataset)} passages | {len(LAYERS)} layers: {LAYERS}")

In [ ]:
# Cost before committing: one sample gives the per-sample shape.
probe = mapper.map(dataset, LAYERS, batch_size=1, progress=False).summary()
per_sample = probe["elements"].sum() * 4
print(f"{per_sample / 1024:.0f} KB per passage")
print(f"{per_sample * len(dataset) / 1024**2:.0f} MB for {len(dataset)} passages")
print(f"{per_sample * 100_000 / 1024**3:.1f} GB for 100k passages")

In [ ]:
# path= streams to HDF5 instead of RAM: memory stays flat in the sample count,
# and the file outlives the session.
store = mapper.map(dataset, LAYERS, CACHE, batch_size=16)
print(f"file on disk: {CACHE.stat().st_size / 1024**2:.1f} MB")
store.metadata

In [ ]:
store.summary()

In [ ]:
# Reads come off disk one sample at a time; nothing is held in RAM.
sample = store[0]
print(store.sample_ids[0], "->", len(sample.activations), "layers,",
      tuple(sample.activations[0].tensor.shape), "each\n")

# Residual stream norm grows with depth - the usual GPT-2 picture.
mask = dataset.encoded["attention_mask"][0].bool()
for act in sample.activations:
    real = act.tensor[mask]  # skip padding positions
    print(f"  {act.layer_name:<6} mean L2 norm = {real.norm(dim=-1).mean():6.2f}")

In [ ]:
store.close()

# Reopening verifies the cache matches the dataset it was built from: ids are
# hashed in order, so a reordered or different dataset is rejected here rather
# than silently misaligning.
reloaded = H5ActivationStore.load(CACHE, [f"wiki_{i:04d}" for i in range(len(dataset))])
print(f"reloaded {len(reloaded)} samples, {len(reloaded.layer_names)} layers")
print(reloaded.metadata)  # travels with the file
reloaded.close()

CACHE.unlink(missing_ok=True)  # drop this line to keep the cache